# 05 — Final Load Preparation (Tableau-ready flat file)

**Owner:** KPI & Final Load Owner
**Input:** `data/processed/online_retail_II_clean.csv`
**Output:** `data/processed/tableau_final_load.csv` — the single flat file Tableau connects to
**Rubric link:** Dashboard & Visualization (20 marks)

## Why a flat file
Tableau Public works best with a single wide CSV. We pre-compute the KPI columns so the dashboard stays fast and the viz lead doesn't fight calculated fields.

## Output schema

The final CSV has:
- All columns from the cleaned data
- Per-customer columns: `rfm_recency`, `rfm_frequency`, `rfm_monetary`, `rfm_segment`
- Per-invoice columns: `invoice_total`
- Flags: `is_top_20pct_customer`

This lets the Viz lead slice freely on any dimension.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../data/processed/online_retail_II_clean.csv", parse_dates=["invoicedate"])
print(df.shape)

## RFM segmentation

We compute RFM only on registered customers (guests can't be tracked).
- **Recency:** days since last purchase, as of the last date in the dataset
- **Frequency:** number of unique invoices
- **Monetary:** net revenue (product-only, returns subtracted)

Each is binned into quintiles (1–5). Segment label = concatenation (e.g. `R5F5M5` = "Champions").

In [ ]:
reg = df[df["is_registered"] & df["is_product"]].copy()
asof = reg["invoicedate"].max()
print(f"As-of date for RFM: {asof}")

rfm = (reg
       .groupby("customer_id")
       .agg(recency_days=("invoicedate", lambda s: (asof - s.max()).days),
            frequency=("invoice", "nunique"),
            monetary=("line_revenue", "sum"))
       .reset_index())

# Quintile scores — higher = better for F and M; reverse for R
rfm["R"] = pd.qcut(rfm["recency_days"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
rfm["rfm_cell"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)

def segment_name(row):
    r, f, m = row["R"], row["F"], row["M"]
    if r >= 4 and f >= 4 and m >= 4: return "Champions"
    if r >= 3 and f >= 4:            return "Loyal"
    if r >= 4 and f <= 2:            return "New"
    if r <= 2 and f >= 4:            return "At Risk"
    if r <= 2 and f <= 2:            return "Lost"
    return "Others"

rfm["rfm_segment"] = rfm.apply(segment_name, axis=1)
rfm["rfm_segment"].value_counts()

## Top-20% customer flag

In [ ]:
cust_rev = reg.groupby("customer_id")["line_revenue"].sum().sort_values(ascending=False)
threshold = cust_rev.quantile(0.80)
rfm["is_top_20pct_customer"] = rfm["monetary"] >= threshold
print(f"Top-20% threshold (monetary): £{threshold:.2f}")
print(f"Top-20% customers: {rfm['is_top_20pct_customer'].sum()} / {len(rfm)}")

## Per-invoice totals

In [ ]:
inv_totals = (df[df["is_product"] & ~df["is_return"]]
              .groupby("invoice")["line_revenue"].sum()
              .rename("invoice_total").reset_index())

## Join everything back into one wide file

In [ ]:
final = df.merge(rfm[["customer_id","recency_days","frequency","monetary",
                      "R","F","M","rfm_cell","rfm_segment","is_top_20pct_customer"]],
                 on="customer_id", how="left")
final = final.merge(inv_totals, on="invoice", how="left")
print("Final shape:", final.shape)
print("Columns:", list(final.columns))

In [ ]:
OUT = Path("../data/processed/tableau_final_load.csv")
final.to_csv(OUT, index=False)
print(f"Wrote {OUT}")

## Hand-off to Viz lead

- Data source file: `data/processed/tableau_final_load.csv`
- Key fields for dashboard:
  - Time: `invoicedate`, `invoice_year_month`
  - Geo: `country_clean`
  - Customer: `rfm_segment`, `is_top_20pct_customer`, `is_registered`
  - Product: `stockcode`, `description`, `product_category_hint`
  - Metrics: `line_revenue`, `invoice_total`, `recency_days`, `frequency`, `monetary`
- Filters to expose: Country, Date range, RFM segment, Registered flag
